# FUNGI-TUBE: Segmented Regression (SR) Featurization Pipeline

**Purpose:** Extract kinetic features from curated FUNGI-TUBE time-series data using piecewise-linear (segmented) regression, then compile results across all runs.

## Pipeline Overview

This notebook performs three tasks:

1. **Curated Time-Series Visualization** — Plots all curated signal columns across replicate files to visually inspect raw data quality and inter-replicate consistency.

2. **Segmented Regression Visualization** — For each signal family, reconstructs piecewise-linear fits from pre-computed kinetic feature files (`kinetic_features_*.csv`) and overlays them on the curated time-series. Each replicate is assigned a consistent color across raw data and fitted segments. Breakpoints and segment slopes/intercepts are extracted from the kinetic feature files and used to reconstruct the fit lines.

3. **Feature Compilation** — Gathers all individual `kinetic_features_*_datalog.csv` files and compiles them into a single `kinetic_features_ALL_compiled.csv` table with a `File` column identifying each run.

## Feature Architecture

The SR approach fits piecewise-linear models with **data-driven breakpoints** to each signal. The number of segments varies by signal family (1–6 segments for VOC, 2–3 for most others). Features per signal include:
- **Segment-level statistics:** median, min, max per segment
- **Fit parameters:** slope, intercept per segment
- **Breakpoint timing:** `t_break1`, `t_break2`, etc.
- **Segment durations:** hours per segment
- **Fit quality:** `fit_ok` flag

## Input Requirements

- `curated_features_*.csv` — Processed time-series files (one per run) with an `Hours` column and all signal columns
- `kinetic_features_*.csv` — Pre-computed segmented regression features (one per run), generated by `FUNGI-TUBE_feature_processing.ipynb`

## Inputs and Outputs

Set `INPUT_DIR` in the visualization cells and `DATA_DIR` in the compilation cell before running. The visualization cells read curated and kinetic feature CSVs from the configured directory, while the compilation cell writes `kinetic_features_ALL_compiled.csv` back to `DATA_DIR`.

## Configuration

Edit `INPUT_DIR` and `DATA_DIR` in the cells below to point to your data folder. All other parameters (smoothing, plot limits, overlay toggles) are configurable at the top of each cell.

In [ ]:
# === Plot all curated features across replicate files (inline) ===
# Configuration
INPUT_DIR = "/Users/jacobwiniski/Desktop/Data"                 # folder containing curated_features_*.csv
FILENAME_PATTERN = "curated_features_*.csv"
XCOL = "Hours"                          # x-axis column if present, else index is used
COLUMN_MODE = "intersection"            # "intersection" or "union"
ROLLING = 0                             # rolling window (samples) for optional smoothing (0 = off)
MAX_PLOTS = None                        # limit number of features to plot (None = all)
CURATED_FEATURE_WHITELIST = []          # e.g., ["Capacitance_detrended","CO2ppm_relChange",...]

# Metadata columns excluded from plotting
EXCLUDE_COLS = {
    XCOL, "DateTime", "timestamp", "time", "Time", "Datetime",
    # add known metadata columns if needed
}

# ---- Data loading and plotting workflow ----
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_files(input_dir: str, pattern: str):
    p = Path(input_dir).expanduser().resolve()
    return sorted([f for f in p.glob(pattern) if f.is_file()])

def load_df(path: Path):
    df = pd.read_csv(path)
    # Try to parse DateTime if present
    if "DateTime" in df.columns:
        try:
            df["DateTime"] = pd.to_datetime(df["DateTime"])
        except Exception:
            pass
    # Coerce numerics where possible (leave DateTime as-is)
    for c in df.columns:
        if c != "DateTime":
            df[c] = pd.to_numeric(df[c], errors="ignore")
    # Determine usable x
    x_used = XCOL if XCOL in df.columns else None
    return df, x_used

def numeric_columns(df: pd.DataFrame, exclude=set()):
    cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return [c for c in cols if c not in exclude]

def compute_feature_list(dfs_info, column_mode="intersection"):
    """
    dfs_info: list of tuples (name, df, x_used)
    """
    if CURATED_FEATURE_WHITELIST:
        return list(CURATED_FEATURE_WHITELIST)

    sets = []
    for (_, df, _) in dfs_info:
        sets.append(set(numeric_columns(df, EXCLUDE_COLS)))

    if not sets:
        return []

    if column_mode == "union":
        feats = sorted(set().union(*sets))
    else:
        feats = sorted(set.intersection(*sets))
    return feats

def maybe_smooth(series: pd.Series, window: int):
    if window and window > 1:
        return series.rolling(window=window, min_periods=max(1, window//2)).mean()
    return series

# 1) Gather files
files = find_files(INPUT_DIR, FILENAME_PATTERN)
if not files:
    raise FileNotFoundError(f"No files found in {INPUT_DIR!r} matching {FILENAME_PATTERN!r}")

# 2) Load dataframes
dfs_info = []  # (replicate_label, df, x_used)
for f in files:
    df, x_used = load_df(f)
    # a readable label: strip extension
    label = f.stem
    dfs_info.append((label, df, x_used))

# 3) Determine features to plot
features = compute_feature_list(dfs_info, column_mode=COLUMN_MODE)
if not features:
    raise RuntimeError("No plot-worthy features found. "
                       "Try setting COLUMN_MODE='union' or filling CURATED_FEATURE_WHITELIST.")

if MAX_PLOTS is not None:
    features = features[:MAX_PLOTS]

# 4) Plot each feature with all replicates overlaid
print(f"Found {len(files)} replicate files and {len(features)} curated features.")
for feat in features:
    plt.figure(figsize=(10, 5))
    has_any = False

    for (label, df, x_used) in dfs_info:
        if feat not in df.columns:
            # For union mode, some files may not have a given feature
            continue
        y = maybe_smooth(df[feat], ROLLING)

        if x_used is not None:
            x = df[x_used]
        else:
            x = np.arange(len(df))

        # Drop NaN pairs
        mask = pd.notna(x) & pd.notna(y)
        if mask.sum() == 0:
            continue

        plt.plot(x[mask], y[mask], linewidth=1.2, label=label)
        has_any = True

    if not has_any:
        plt.close()
        continue

    # Labels & cosmetics
    plt.title(feat)
    plt.xlabel(XCOL if any(xu is not None for (_, _, xu) in dfs_info) else "Index")
    plt.ylabel(feat)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()

# Print a compact processing summary
print("Done.")

In [ ]:
# === Plot ALL segmented features with consistent per-replicate colors ===
# Scans INPUT_DIR for kinetic_features_*.csv and curated_features_*.csv, pairs by suffix,
# reconstructs piecewise-linear fits, and renders one plot per discovered group
# with all replicates overlaid. Each replicate uses a single consistent color
# across its raw overlay and all segment pieces.

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

# Configuration
INPUT_DIR        = "/Users/jacobwiniski/Desktop/Data"
CURATED_PATTERN  = "curated_features_*.csv"
KINETIC_PATTERN  = "kinetic_features_*.csv"
XCOL             = "Hours"

OVERLAY_RAW      = False
RAW_ALPHA        = 0.35
ROLLING_RAW      = 0

RAW_OVERLAY_ALIASES = {
    "cap":     ["Capacitance_detrended", "Capacitance_rate"],
    "voc":     ["VOC_RH_relChange", "VOC_Raw_relChange", "VOCppm_relChange"],
    "co2_rel": ["CO2ppm_relChange_tempDetrended", "CO2ppm_relChange"],
    "co2":     ["CO2ppm_relChange_tempDetrended", "CO2ppm_relChange"],
    "ir":      ["IR_Delta_relChange"],
    "mbi":     ["MetabolicBalanceIndex"],
    "rh":      ["RH_relChange", "RelativeHumidity_relChange"],
}

MAX_GROUPS = None
SHOW_ONLY_GROUPS = None

# Helpers
def find_files(input_dir: str, pattern: str):
    p = Path(input_dir).expanduser().resolve()
    return sorted([f for f in p.glob(pattern) if f.is_file()])

def key_from_name(path: Path):
    stem = path.stem
    stem = re.sub(r"^(curated_features_|kinetic_features_)", "", stem)
    return stem

def load_curated(path: Path, xcol: str):
    df = pd.read_csv(path)
    if "DateTime" in df.columns:
        try:
            df["DateTime"] = pd.to_datetime(df["DateTime"])
        except Exception:
            pass
    for c in df.columns:
        if c != "DateTime":
            df[c] = pd.to_numeric(df[c], errors="ignore")
    x_used = xcol if xcol in df.columns else None
    return df, x_used

def maybe_smooth(series: pd.Series, window: int):
    if window and window > 1:
        return series.rolling(window=window, min_periods=max(1, window//2)).mean()
    return series

def reconstruct_segments(row: pd.Series, group: str, t_end: float):
    break_cols = []
    for i in range(1, 10):
        for cand in (f"{group}_t_break{i}", f"{group}_break{i}_h", f"{group}_break{i}", f"{group}_tbreak{i}"):
            if cand in row.index:
                break_cols.append(cand)
                break
    breaks = []
    for bc in break_cols:
        val = row[bc]
        if pd.notna(val):
            try:
                breaks.append(float(val))
            except Exception:
                pass
    breaks = sorted({b for b in breaks if 0 <= b <= t_end})

    slope_cols = [c for c in row.index if re.fullmatch(fr"{re.escape(group)}_slope_seg\d+", c)]
    intc_cols  = [c for c in row.index if re.fullmatch(fr"{re.escape(group)}_intercept_seg\d+", c)]
    slope_cols.sort(key=lambda s: int(re.search(r"(\d+)$", s).group(1)))
    intc_cols.sort(key=lambda s: int(re.search(r"(\d+)$", s).group(1)))

    nsegs = min(len(slope_cols), len(intc_cols))
    if nsegs == 0:
        return []

    t_nodes = [0.0] + breaks + [float(t_end)]
    t_nodes = [t_nodes[i] for i in range(len(t_nodes)) if (i == 0 or t_nodes[i] >= t_nodes[i-1])]

    segs = []
    for i in range(nsegs):
        t0 = t_nodes[i]
        t1 = t_nodes[i+1] if i+1 < len(t_nodes) else t_end
        if t1 <= t0:
            continue
        try:
            slope = float(row[slope_cols[i]])
            intercept = float(row[intc_cols[i]])
        except Exception:
            continue
        segs.append({"t0": t0, "t1": t1, "slope": slope, "intercept": intercept})
    return segs

def discover_groups(kdf: pd.DataFrame):
    groups = set()
    for c in kdf.columns:
        m = re.match(r"^(.+?)_(slope|intercept)_seg\d+$", c)
        if m:
            groups.add(m.group(1))
    return sorted(groups)

def choose_overlay_column(curated_df: pd.DataFrame, group: str):
    if group in RAW_OVERLAY_ALIASES:
        for cand in RAW_OVERLAY_ALIASES[group]:
            if cand in curated_df.columns:
                return cand
    for cand in (group, group.upper(), group.lower(),
                 f"{group}_relChange", f"{group}_detrended"):
        if cand in curated_df.columns:
            return cand
    return None

# Load and pair files
curated_files = find_files(INPUT_DIR, CURATED_PATTERN)
kinetic_files = find_files(INPUT_DIR, KINETIC_PATTERN)
if not kinetic_files:
    raise FileNotFoundError(f"No kinetic files found in {INPUT_DIR} matching {KINETIC_PATTERN}")

curated_map = {key_from_name(cf): cf for cf in curated_files}
kinetic_map  = {key_from_name(kf): kf for kf in kinetic_files}

replicates = []
all_groups = set()

for key, kpath in kinetic_map.items():
    kdf = pd.read_csv(kpath)
    krow = kdf.iloc[0]
    groups_here = discover_groups(kdf)
    all_groups.update(groups_here)

    cdf, x_used, t_end = None, None, None
    cpath = curated_map.get(key)
    if cpath is not None:
        cdf, x_used = load_curated(cpath, XCOL)
        t_end = float(pd.to_numeric(cdf[x_used], errors="coerce").max()) if x_used else float(len(cdf) - 1)
    else:
        t_end = float(krow.get("cap_seg1_duration", 1.0))

    replicates.append({
        "key": key,
        "kinetic_row": krow,
        "curated": cdf,
        "x_used": x_used,
        "t_end": t_end,
        "groups": groups_here
    })

all_groups = sorted(all_groups)
if SHOW_ONLY_GROUPS:
    all_groups = [g for g in all_groups if g in SHOW_ONLY_GROUPS]
if MAX_GROUPS is not None:
    all_groups = all_groups[:MAX_GROUPS]

print(f"Discovered groups: {all_groups}")
print(f"Replicates: {len(replicates)}; curated matched: {sum(r['curated'] is not None for r in replicates)}")

# Consistent colors per replicate
# Build a color mapping once; reuse for raw + all segments of that replicate.
base_colors = list(plt.cm.tab20.colors) + list(plt.cm.Set3.colors) + list(plt.cm.Paired.colors)
color_cycle = itertools.cycle(base_colors)
rep_colors = {}
for rep in replicates:
    rep_colors[rep["key"]] = next(color_cycle)

# Plot every discovered group
for group in all_groups:
    plt.figure(figsize=(10, 5))
    drew_any = False

    # Raw overlays (use same replicate color, lighter alpha)
    if OVERLAY_RAW:
        for rep in replicates:
            cdf, x_used = rep["curated"], rep["x_used"]
            if cdf is None:
                continue

            col = choose_overlay_column(cdf, group)
            if col is None and group in ("co2", "co2_rel"):
                if "CO2ppm_relChange_tempDetrended" in cdf.columns:
                    col = "CO2ppm_relChange_tempDetrended"
                elif "CO2ppm_relChange" in cdf.columns:
                    col = "CO2ppm_relChange"
            if col is None or col not in cdf.columns:
                continue

            x = cdf[x_used] if x_used else np.arange(len(cdf))
            y = cdf[col]
            if ROLLING_RAW and ROLLING_RAW > 1:
                y = maybe_smooth(y, ROLLING_RAW)
            mask = pd.notna(x) & pd.notna(y)
            if not mask.sum():
                continue

            plt.plot(
                x[mask], y[mask],
                color=rep_colors[rep["key"]],
                alpha=RAW_ALPHA, linewidth=1.0,
                label=f"{rep['key']} raw"
            )

    # Segmented fits (reuse same color)
    for rep in replicates:
        if group not in rep["groups"]:
            continue
        row, t_end = rep["kinetic_row"], rep["t_end"]
        segs = reconstruct_segments(row, group, t_end)
        if not segs:
            continue

        label_added = False
        for seg in segs:
            xs = np.array([seg["t0"], seg["t1"]], dtype=float)
            ys = seg["intercept"] + seg["slope"] * xs
            plt.plot(
                xs, ys,
                color=rep_colors[rep["key"]],
                linewidth=2.0,
                label=f"{rep['key']} fit" if not label_added else None
            )
            label_added = True
        drew_any = True

    if not drew_any:
        plt.close()
        continue

    # Axis titles/labels
    ylabel = group
    if OVERLAY_RAW:
        for rep in replicates:
            cdf = rep["curated"]
            if cdf is None:
                continue
            col_guess = choose_overlay_column(cdf, group)
            if col_guess:
                ylabel = col_guess
                break

    plt.title(f"Segmented linear fit: {group}")
    plt.xlabel(XCOL)
    plt.ylabel(ylabel)
    plt.legend(loc="best", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

print("Done plotting all discovered groups with consistent per-replicate colors.")

In [ ]:
# Compile all "kinetic_features*"_datalog.csv files into one table with a 'File' column.
# 'File' is the text between 'kinetic_features_' and '_datalog.csv' (e.g., "Run_01").

from pathlib import Path
import pandas as pd
import re

# Configuration
DATA_DIR = Path("/Users/jacobwiniski/Desktop/Data") 
OUTPUT_CSV = DATA_DIR / "kinetic_features_ALL_compiled.csv"

# Gather and load files
pattern = "kinetic_features*_datalog.csv"
files = sorted(DATA_DIR.glob(pattern))

df_list = []
rx = re.compile(r"kinetic_features_(.+?)_datalog\.csv$", re.IGNORECASE)

for f in files:
    try:
        # Extract the run/file label between kinetic_features_ and _datalog.csv
        m = rx.search(f.name)
        file_label = m.group(1) if m else f.stem  # fallback to stem if no match

        df = pd.read_csv(f)
        df.insert(0, "File", file_label)  # put 'File' as first column
        df_list.append(df)
    except Exception as e:
        print(f"[WARN] Skipping {f.name}: {e}")

if not df_list:
    raise SystemExit("No matching kinetic_features files found. Check DATA_DIR and pattern.")

# Use outer join to accommodate any column mismatches across runs
compiled = pd.concat(df_list, ignore_index=True, join="outer")

# Save and preview compiled features
compiled.to_csv(OUTPUT_CSV, index=False)
print(f"Compiled {len(df_list)} file(s) into: {OUTPUT_CSV}")
display(compiled.head())